In [156]:
import pandas as pd

people = pd.read_sas("/Users/samkurlender/Personal Projects/Youth Migration from Long Island/sas_pny/psam_p36.sas7bdat", encoding="latin1")

use = ["PUMA", "AGEP", "MIG", "MIGSP", "MIGPUMA", "ADJINC", "WAGP", "SEMP",
       "PINCP", "SCHL", "ESR", "SCH", "PWGTP"] + [f"PWGTP{i}" for i in range(1, 81)]
sixteen_to_twenty = pd.read_csv("/Users/samkurlender/Personal Projects/Youth Migration from Long Island/csv_pny (16-20)/psam_p36.csv", usecols=use, dtype={"PUMA": str})
sixteen_to_twenty["PUMA"] = sixteen_to_twenty["PUMA"].str.strip().str.zfill(5)


/var/folders/fc/pdzs20394zjgfkqhywq86g940000gn/T/ipykernel_10200/3641516707.py:7: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  sixteen_to_twenty = pd.read_csv("/Users/samkurlender/Personal Projects/Youth Migration from Long Island/csv_pny (16-20)/psam_p36.csv", usecols=use, dtype={"PUMA": str})


## Cleaning Data

In [157]:
for col in ["STATE", "MIG", "MIGSP", "MIGPUMA", "PWGTP"]:
    people[col] = pd.to_numeric(people[col], errors = "coerce")

for col in ["ADJINC", "WAGP", "SEMP", "PWGTP1", "PWGTP80"]: 
    sixteen_to_twenty[col] = pd.to_numeric(sixteen_to_twenty[col], errors = "coerce")


people.head()

# Decode byte strings like b'04310' into ordinary strings like '04310'
people["PUMA"] = (
    people["PUMA"]
    .astype("string")
    .str.replace(r"^b['\"]|['\"]$", "", regex=True)
    .str.strip()
    .str.zfill(5)
)


## Bringing in crosswalk data

In [158]:
crosswalk = pd.read_excel("/Users/samkurlender/Personal Projects/Youth Migration from Long Island/county_migpuma_pwpuma_2022.xls")

crosswalk

,State of Residence (ST),County of Residence,Place of Work State (PWSTATE2) or Migration State (MIGPLAC1),PWPUMA00 or MIGPUMA1
0,1,1,1,1700
1,1,3,1,2700
2,1,5,1,2500
3,1,7,1,1100
4,1,9,1,900
...,...,...,...,...
3217,72,145,72,1600
3218,72,147,72,800
3219,72,149,72,600
3220,72,151,72,800


In [159]:
import pandas as pd
import numpy as np

# ---------- 1. Convert relevant variables to numeric (both files) ----------
for df in (people, sixteen_to_twenty):
    for col in ["AGEP", "MIG", "MIGSP", "MIGPUMA", "PWGTP"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        
# ACS has 80 weights, converting them to numeric in both files
REPS = [f"PWGTP{i}" for i in range(1, 81)]

for df in (nyc_movers, nyc_movers1620):
    for c in REPS + ["PWGTP"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# ---------- 2. Maps ----------
# Origin codes (MIGPUMA, one year ago): the same in both files
origin_map = {3200: "Nassau County", 3300: "Suffolk County"}

# Destination (current PUMA prefix): DIFFERENT in each file
dest_map_24 = {"041": "Manhattan", "042": "Bronx", "043": "Brooklyn",
               "044": "Queens", "045": "Staten Island"}              # 2020-vintage PUMAs (2024 file)

dest_map_1620 = {"037": "Bronx", "038": "Manhattan", "039": "Staten Island",
                 "040": "Brooklyn", "041": "Queens"}                 # 2010-vintage PUMAs (2016-2020 file)

BORO_COLS = ["Manhattan", "Bronx", "Brooklyn", "Queens", "Staten Island"]
LI_ROWS = ["Nassau County", "Suffolk County"]

# ---------- 3. One builder, used for both files ----------
def decode_puma(s):
    # Handles bytes (b'04310') or text; returns zero-padded 5-character strings
    return s.apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else str(x)).str.strip().str.zfill(5)

def build_nyc_movers(df, dest_map):
    mask = (
        df["AGEP"].between(18, 35)
        & (df["MIG"] == 3)
        & (df["MIGSP"] == 36)
        & df["MIGPUMA"].isin(origin_map)          # lived in Nassau/Suffolk a year ago
    )
    movers = df.loc[mask].copy()
    movers["origin"] = movers["MIGPUMA"].map(origin_map)
    movers["destination"] = decode_puma(movers["PUMA"]).str[:3].map(dest_map)
    return movers.dropna(subset=["destination"]).copy()    # keep only the five boroughs

nyc_movers     = build_nyc_movers(people, dest_map_24)            # 2024
nyc_movers1620 = build_nyc_movers(sixteen_to_twenty, dest_map_1620)  # 2016-2020

# ---------- 4. Tables ----------
def show_tables(movers, title):
    weighted = pd.pivot_table(
        movers, index="origin", columns="destination",
        values="PWGTP", aggfunc="sum", fill_value=0
    ).reindex(index=LI_ROWS, columns=BORO_COLS, fill_value=0)

    counts = pd.crosstab(movers["origin"], movers["destination"]).reindex(
        index=LI_ROWS, columns=BORO_COLS, fill_value=0)

    print(f"{title}: estimated Nassau/Suffolk-origin young adults living in NYC")
    display(weighted.round(0))
    print("Unweighted respondent counts:")
    display(counts)
    print("Total weighted:", round(weighted.to_numpy().sum()), "| records:", len(movers))
    print()

show_tables(nyc_movers, "2024")
show_tables(nyc_movers1620, "2016-2020 (annual average)")

2024: estimated Nassau/Suffolk-origin young adults living in NYC


destination,Manhattan,Bronx,Brooklyn,Queens,Staten Island
origin,,,,,
Nassau County,1727.0,215.0,95.0,1543.0,80.0
Suffolk County,550.0,0.0,2503.0,795.0,71.0


Unweighted respondent counts:


destination,Manhattan,Bronx,Brooklyn,Queens,Staten Island
origin,,,,,
Nassau County,13,3,2,13,1
Suffolk County,7,0,13,13,2


Total weighted: 7579 | records: 67

2016-2020 (annual average): estimated Nassau/Suffolk-origin young adults living in NYC


destination,Manhattan,Bronx,Brooklyn,Queens,Staten Island
origin,,,,,
Nassau County,2145,188,1108,3163,53
Suffolk County,1579,88,794,1237,98


Unweighted respondent counts:


destination,Manhattan,Bronx,Brooklyn,Queens,Staten Island
origin,,,,,
Nassau County,96,13,62,144,2
Suffolk County,98,7,35,65,3


Total weighted: 10453 | records: 525



In [160]:
# import pandas as pd
# import numpy as np

# # Convert relevant variables to numeric
# for col in ["AGEP", "MIG", "MIGSP", "MIGPUMA", "PWGTP"]:
#     people[col] = pd.to_numeric(people[col], errors="coerce")
#     sixteen_to_twenty[col] = pd.to_numeric(sixteen_to_twenty[col], errors="coerce")


# # Identify young adults who lived in Nassau/Suffolk-associated
# # Migration PUMAs one year ago and moved within the US
# mask = (
#     people["AGEP"].between(18, 35)
#     & (people["MIG"] == 3)
#     & (people["MIGSP"] == 36)
#     & people["MIGPUMA"].isin([3200, 3300])
# )

# mask = (
#     sixteen_to_twenty["AGEP"].between(18, 35)
#     & (sixteen_to_twenty["MIG"] == 3)
#     & (sixteen_to_twenty["MIGSP"] == 36)
#     & sixteen_to_twenty["MIGPUMA"].isin([3200, 3300])
# )

# LI_movers24 = people.loc[mask].copy()
# LI_movers1620 = sixteen_to_twenty.loc[mask].copy()

# LI_movers24["origin"] = LI_movers24["MIGPUMA"].map({
#     3200: "Nassau County",
#     3300: "Suffolk County"
# })

# dest_map_1620 = {"037": "Bronx", "038": "Manhattan", "039": "Staten Island",
#                  "040": "Brooklyn", "041": "Queens"}

# LI_movers1620 = {"032": "Nassau County", "033": "Suffolk County"}   # same in both vintages

# # Decode current PUMA values, e.g. b'04310' -> '04310'
# puma_codes = LI_movers24["PUMA"].apply(
#     lambda x: x.decode("utf-8") if isinstance(x, bytes) else str(x)
# ).str.strip().str.zfill(5)

# destination_map = {
#     "041": "Manhattan",
#     "042": "Bronx",
#     "043": "Brooklyn",
#     "044": "Queens",
#     "045": "Staten Island"
# }



# LI_movers24["destination"] = puma_codes.str[:3].map(destination_map)
# LI_movers1620["destination"] = puma_codes.str[:3].map(destination_map)


# # Keep only people whose current destination is one of the five boroughs
# nyc_movers = LI_movers24.dropna(subset=["destination"]).copy()
# nyc_movers1620 = LI_movers1620.dropna(subset=["destination"]).copy()


# # Table of unweighted records and weighted estimates
# migration_table = pd.pivot_table(
#     nyc_movers,
#     index="origin",
#     columns="destination",
#     values="PWGTP",
#     aggfunc="sum",
#     fill_value=0
# ).reindex(
#     index=["Nassau County", "Suffolk County"],
#     columns=["Manhattan", "Bronx", "Brooklyn", "Queens", "Staten Island"],
#     fill_value=0
# )

# print("Estimated Nassau/Suffolk-origin young adults currently living in NYC")
# display(migration_table.round(0))

# print("Unweighted respondent counts:")
# display(
#     pd.crosstab(nyc_movers["origin"], nyc_movers["destination"])
#     .reindex(
#         index=["Nassau County", "Suffolk County"],
#         columns=["Manhattan", "Bronx", "Brooklyn", "Queens", "Staten Island"],
#         fill_value=0
#     )
# )

In [161]:
print("The estimated number of 18-35 year olds moving from Nassau County to one of the 5 boroughs: ", sum(migration_table.loc["Nassau County"]))
print("The estimated number of 18-35 year olds moving from Suffolk County to one of the 5 boroughs: ", sum(migration_table.loc["Suffolk County"]))

The estimated number of 18-35 year olds moving from Nassau County to one of the 5 boroughs:  3660.0
The estimated number of 18-35 year olds moving from Suffolk County to one of the 5 boroughs:  3919.0


#### Standard Error

In [162]:
WCOL  = "PWGTP"                                  # main person weight
RCOLS = [f"PWGTP{i}" for i in range(1, 81)]     # the 80 replicate weights (same list as REPS)

def w_count(df, w):
    """Weighted headcount: add up the weights in column w."""
    return df[w].sum()

def with_se(df, stat, **kwargs):
    """Run a statistic with the main weight, then with each replicate weight,
    and turn the spread of the replicate results into a standard error."""
    est  = stat(df, WCOL, **kwargs)                              # the estimate
    reps = np.array([stat(df, r, **kwargs) for r in RCOLS])      # 80 replicate estimates
    se   = np.sqrt(4 / 80 * ((reps - est) ** 2).sum())           # Census (ACS) formula
    return {"estimate": est, "se": se, "moe90": 1.645 * se}

In [163]:
def compare_periods(old, new, by = None):
    groups = [(None, old, new)] if by is None else [
        (k, old[old[by] == k], new[new[by] == k]) for k in old[by].dropna().unique()]
    for k, o, n in groups:
        a, b = with_se(o, w_count), with_se(n, w_count)
        diff = b["estimate"] - a["estimate"]
        se = np.sqrt(a["se"] ** 2 + b["se"] ** 2)
        label = "All" if k is None else k
        print(f"{label:15s} 2016-20: {a['estimate']:7.0f}  2024: {b['estimate']:7.0f}  "
              f"change: {diff:7.0f}  SE: {se:6.0f}  z: {diff/se:5.2f}")

compare_periods(nyc_movers1620, nyc_movers)
compare_periods(nyc_movers1620, nyc_movers, by = "origin")

All             2016-20:   10453  2024:    7579  change:   -2874  SE:   1556  z: -1.85
Nassau County   2016-20:    6657  2024:    3660  change:   -2997  SE:    984  z: -3.04
Suffolk County  2016-20:    3796  2024:    3919  change:     123  SE:   1205  z:  0.10


### Local Spending Impact

local spending impact = movers × after-tax income × share spent × share spent locally × multiplier

In [186]:
our_pop = nyc_movers[nyc_movers['destination'].isin(destination_map.values())]

In [187]:
# inflation adjustment factor
our_pop["ADJINC"] = pd.to_numeric(our_pop["ADJINC"], errors="coerce")
f = our_pop["ADJINC"] / 1_000_000
our_pop["Earnings"] = (our_pop['WAGP'].fillna(0) + our_pop['SEMP'].fillna(0)) * f

In [188]:
print((our_pop["SEMP"] < 0).sum(), "people with negative SEMP")
print((our_pop["Earnings"] == 0).sum(), "people with zero total earnings")

w = our_pop["PWGTP"]
print("Weighted share with zero earnings:", w[our_pop["Earnings"] == 0].sum() / w.sum())

0 people with negative SEMP
10 people with zero total earnings
Weighted share with zero earnings: 0.10740203193033382


In [189]:
# looking at earners only - removing the 10 people who had 0 earnings listed

earners = our_pop[our_pop["Earnings"] != 0]

print("Mean, all movers:", our_pop["Earnings"].mean())
print("Mean, earners:", earners["Earnings"].mean())

Mean, all movers: 64086.51977611941
Mean, earners: 75329.76885964912


In [190]:
# verifying the non-earners
zero = our_pop[our_pop["Earnings"] == 0]
print(zero.groupby("ESR")["PWGTP"].sum())
# 3 is unemployed, 6 is not in the labor force

print(zero.groupby("SCH")["PWGTP"].sum())
# 1 is no, 2 is yes public, 3 is yes private

ESR
3     51.0
6    763.0
Name: PWGTP, dtype: float64
SCH
1    431.0
2     76.0
3    307.0
Name: PWGTP, dtype: float64


In [191]:
zero["ESR"]
percents = pd.to_numeric(zero["ESR"], errors = "coerce")

percents

2307      6
5234      6
7407      6
11764     6
30126     6
49167     6
52785     3
162111    6
177740    6
204078    6
Name: ESR, dtype: int64

In [192]:
zero[:]['PINCP']

2307      2400.0
5234         0.0
7407         0.0
11764     2400.0
30126        0.0
49167        0.0
52785      200.0
162111       0.0
177740     400.0
204078       0.0
Name: PINCP, dtype: float64

In [193]:
edges = [0, 25_000, 50_000, 100_000, 200_000]          # 5 edges -> 6 earnings bands

# ILLUSTRATIVE effective tax rate per band. REPLACE with sourced values.
rates = np.array([0.00, 0.10, 0.18, 0.25, 0.32, 0.37])

def add_after_tax(df):
    earn = df["Earnings"].clip(lower=0)                 # treat business losses as zero
    band = np.digitize(earn, edges, right=True)         # which band each person falls in
    df["AfterTax"] = earn * (1 - rates[band])
    return df

our_pop = add_after_tax(our_pop)

In [194]:
local_rates = {"low (40%)": 0.40, "mid (50%)": 0.50, "high (65%)": 0.65}

income = with_se(our_pop, w_total, value="AfterTax")    # weighted after-tax total, with error
n      = with_se(our_pop, w_count)                      # weighted headcount

rows = []
for name, k in local_rates.items():
    rows.append({
        "scenario": name,
        "direct_local_spending_$M": income["estimate"] * k / 1e6,
        "moe90_$M": income["moe90"] * k / 1e6,
        "per_person_$": income["estimate"] * k / n["estimate"],
    })

print(pd.DataFrame(rows).round(1).to_string(index=False))

  scenario  direct_local_spending_$M  moe90_$M  per_person_$
 low (40%)                     215.6     135.3       28452.4
 mid (50%)                     269.6     169.1       35565.5
high (65%)                     350.4     219.9       46235.1


### Conclusion

In the 2024 ACS (American Community Survey), the estimated number of 18-35 year olds moving from Nassau County to one of the 5 boroughs was 3,660 (+- 1,400), and 3,920 (+-1,860) moving from Suffolk County within the prior year.

Assuming 40% to 65% of after-tax income is spent locally, and using placeholder tax rates, the 18-35 movers from Long Island to NYC account for roughly 215M to 350M a year of gross direct local spending, which works out to 28K to 46K spending per person. That is gross spending relocated away from Long Islanddd. It's worth noting there's a sizeable margin of error with the estimates, a consequence of a small sample size from a only a single year. 